# Energy Minimization of Lennard-Jones charged particles — Simplex (Nelder–Mead)

*Utrecht University Molecular Modelling courses from the [Bonvin lab](https://bonvinlab.org).*

This notebook performs a simple **energy minimization (EM)** of a 2D system of
Lennard-Jones particles that may also carry a charge (Coulomb interaction).
Starting from a random arrangement, EM walks the system *downhill* on its potential-energy
surface towards a nearby local minimum. The minimizer here is the **downhill simplex
(Nelder–Mead)** method, which uses only **energy evaluations — no forces**.

## Theory in brief

### Lennard-Jones
Written from the squared distance $r^2$, with $Z = (r_{min}^2 / r^2)^3 = (r_{min}/r)^6$:

$$E_{LJ} = \varepsilon\, Z (Z-1)$$

The $Z^2$ term is the steep short-range **repulsion**, the $-Z$ term the weaker long-range
**attraction**; they balance at $r = r_{min}$ (depth $-\varepsilon$). Pairs beyond a cutoff
(`CutOff`) are ignored.

### Coulomb

$$E_{Coul} = \frac{q_a q_b}{\epsilon_r\, r}$$

Like charges repel, unlike charges attract; `Dielec` ($\epsilon_r$) screens the interaction.

### Simplex (Nelder–Mead) minimization
The full set of $2\,n_{atoms}$ particle coordinates is a single point in a high-dimensional
space. The simplex method keeps several trial configurations — the **vertices** of a
simplex — and repeatedly improves its worst (highest-energy) vertex with purely geometric
moves, using only energy values:

* **Reflection** — reflect the highest vertex through the centroid of the others.
* **Expansion** — if the reflection improved the energy, step further in that direction.
* **Contraction** (*"shrimp"* of the highest) — if reflection did not help, pull the highest
  vertex partway toward the centroid.
* **Shrink** (*"shrimp"* toward the lowest) — if nothing helped, contract every vertex toward
  the current best.

`Simplex_step` sets the size of the initial simplex; `FracShrimp1` / `FracShrimp2` and
`FracExpend` control the contraction / expansion. Convergence is declared once the best
energy has improved by less than `deltaE` on `n2conv` steps (counted cumulatively).

Unlike the steepest-descent and conjugate-gradient minimizers (which follow the
**forces**, i.e. the energy gradient), the simplex is **derivative-free** — it needs only
energy values. That makes it simple and robust on rough or noisy energy surfaces, but it
usually needs *more* energy evaluations to reach the minimum.

## 1. Imports

The numerical core uses only the Python **standard library** (`math`, `random`), so it runs
on a bare Python install. **matplotlib** is the one third-party dependency — it draws the
static figures and the trajectory animation (embedded inline as interactive HTML via
`jshtml`). The cell below first **installs matplotlib if it is missing** (handy on Google
Colab), then imports everything; `%matplotlib inline` renders figures inside the notebook.

In [ ]:
# --- Install required packages if missing (e.g. on Google Colab) ---
import importlib.util, subprocess, sys

for pkg in ["matplotlib"]:
    if importlib.util.find_spec(pkg) is None:
        print(f"Installing {pkg} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
    else:
        print(f"{pkg} already available")

from math import sqrt

import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from matplotlib.animation import FuncAnimation
from matplotlib import rc

from random import random, randint, seed

# Show animations inline
rc('animation', html='jshtml')
%matplotlib inline

## 2. Helper functions

Small utilities used throughout:

* `dist` — Euclidean distance between two points.
* `SignR(a, b)` — returns `a` with the sign of `b`. It implements the **minimum-image
  convention** / periodic wrapping used by the energy and boundary routines.
* `charge_color` — purely cosmetic: white for positive charges, dark for negative.

In [ ]:
### distance ###
def dist(A, B):
    return sqrt((A[0]-B[0])**2 + (A[1]-B[1])**2)

### change sign ###
def SignR(a, b):
    if b > 0:
        return a
    else:
        return -a

### colour particles based on charge ###
def charge_color(charge, qat):
    if charge == qat:
        return "#FFFFFF"   # positive
    else:
        return "#333333"   # negative

## 3. Energy functions

The total energy is the sum over all particle pairs of the Lennard-Jones and Coulomb
contributions, using the **nearest image** convention (periodic boundary conditions).
Distances are handled as **squared** distances (`distsquare`) so the inner loop avoids a
`sqrt` for every pair and the cutoff test is cheap; a `sqrt` is taken only where the Coulomb
$1/r$ needs it. A tiny floor on `distsquare` guards against a division by zero if two
particles ever coincide (which simply yields a huge repulsive energy).

The simplex only ever needs this single **total** energy value per configuration.

In [ ]:
# LJ energy from the squared distance
def LJ2(distsquare, epsilon, rmin_exp6):
    Z = (1/distsquare)**3 * rmin_exp6
    return epsilon * Z * (Z - 1)

# classical Coulomb from the squared distance
def Coulomb2(r, dielec, qa, qb):
    return qa*qb / (dielec*sqrt(r))

# Total energy Evdw + Ecoulomb (uses squared distance), with periodic boundary conditions
def Calc_Ene(coord, epsilon, rmin, dielec, cutoffsquare, boxdim, elec=1):
    Ene = 0.0
    rmin_exp6 = rmin**6
    # doubly nested loop over all particle pairs
    for i in range(len(coord)-1):
        for j in range(i+1, len(coord)):
            # squared atomic distance (nearest image)
            distsquare = 0
            for k in range(2):
                tmp = coord[j][k] - coord[i][k]
                halfbox = boxdim[k]/2
                tmp = tmp - SignR(halfbox, tmp-halfbox) - SignR(halfbox, tmp+halfbox)
                distsquare += tmp**2
            distsquare = max(distsquare, 1e-6)   # avoid division by zero
            if distsquare < cutoffsquare:
                qa = coord[i][2]
                qb = coord[j][2]
                Ene += LJ2(distsquare, epsilon, rmin_exp6)
                if elec:
                    Ene += Coulomb2(distsquare, dielec, qa, qb)
    return Ene

## 4. The simplex minimizer

### The idea
A *simplex* is a set of trial points (its **vertices**). Here each vertex is a **complete
configuration** of the `nAtoms` particles, so it lives in a $2\,n_{atoms}$-dimensional space.
The method never uses forces or gradients — it only compares the **energies** of the vertices
and moves the worst (highest-energy) one by simple geometric operations.

Let $\mathbf{x}_h$ be the **highest**-energy vertex and $\mathbf{c}$ the **centroid** of all
the *other* vertices,

$$\mathbf{c} = \frac{1}{N}\sum_{i\neq h}\mathbf{x}_i .$$

Every candidate move has the form $\mathbf{x}_h + f\,(\mathbf{c}-\mathbf{x}_h)$ for a
different factor $f$:

| Move | Geometry | Factor in code |
|---|---|---|
| **Reflection** | $\mathbf{x}_r = 2\mathbf{c}-\mathbf{x}_h$ — mirror the worst point through the centroid | `reflnr = 2` |
| **Expansion** | $\mathbf{x}_e = 2\mathbf{x}_r-\mathbf{c}$ — a further step along a successful reflection | second call, `-FracExpend` |
| **Contraction** | $\mathbf{x}_c = \mathbf{x}_h + 0.8\,(\mathbf{c}-\mathbf{x}_h)$ — pull the worst point most of the way to the centroid | `FracShrimp1` |
| **Shrink** | every vertex $\to \mathbf{x}_i + 0.8\,(\mathbf{x}_{low}-\mathbf{x}_i)$ — contract the whole simplex toward the best | `FracShrimp2` |

Intuitively: reflection walks the simplex *downhill*; expansion accelerates when a direction
keeps paying off; contraction and shrink let the simplex *shrink into* a minimum once it has
bracketed one.

### The code
`make_simplex_coor` builds a displaced configuration; the initial simplex has one vertex per
coordinate direction (the starting configuration nudged by `Simplex_step` along each of the
$2\,n_{atoms}$ axes). The `Simplex` class stores the vertices and offers:

* `energy` — energy of every vertex, plus the indices of the **highest** and **lowest**.
* `Update_Simplex` — replace one vertex.
* `shrimp` — shrink all vertices toward the lowest.
* `boundary` — wrap vertices back into the periodic box.

`Simplex_Reflection` returns, in one call, both the **reflected** ($f=$ `reflnr`) and the
**contracted** ($f=$ `FracShrimp1`) version of the highest vertex. Expansion is obtained by
calling it a second time on the just-reflected vertex with $f = -$`FracExpend`, which
evaluates to $\mathbf{x}_e = 2\mathbf{x}_r-\mathbf{c}$.

In [ ]:
def make_simplex_coor(coord, nr, xory, step):
    """Copy `coord`, displacing coordinate (atom nr, axis xory) by `step`."""
    listnew = []
    for i in range(len(coord)):
        tmplist = []
        for j in range(len(coord[i]) - 1):        # x and y only
            tmp = float(coord[i][j])
            if i == nr and j == xory:
                tmp = tmp + step
            tmplist.append(tmp)
        tmplist.append(coord[i][-1])              # keep the charge
        listnew.append(tmplist)
    return listnew


class Simplex:
    def __init__(self, ac, step):
        # one vertex per coordinate direction: the base config displaced by `step`
        tmpss = []
        for pp in range(len(ac)):
            for qq in range(len(ac[pp]) - 1):
                tmpss.append(make_simplex_coor(ac, pp, qq, step))
        self.points = tmpss
        self.simplex_energy, self.highest_nr, self.lowest_nr = self.energy()

    def energy(self):
        """Energy of every vertex, plus the highest and lowest vertex indices."""
        simplex_energy = [Calc_Ene(v, Epsilon, Rmin, Dielec, CutOffSquare, BoxDim)
                          for v in self.points]
        highest_nr = max(range(len(simplex_energy)), key=lambda k: simplex_energy[k])
        lowest_nr  = min(range(len(simplex_energy)), key=lambda k: simplex_energy[k])
        return simplex_energy, highest_nr, lowest_nr

    def Update_Simplex(self, new, nr):
        self.points[nr] = new

    def shrimp(self, lowest):
        """Shrink every vertex toward the lowest one by FracShrimp2."""
        lowestcoor = self.points[lowest]
        flat = []
        for count, i in enumerate(self.points):
            new = []
            if count != lowest:
                for x1 in range(len(i)):
                    for jj in range(len(i[x1][:-1])):       # x, y
                        new.append(i[x1][jj] + FracShrimp2*(lowestcoor[x1][jj] - i[x1][jj]))
            else:
                for x1 in range(len(i)):
                    for jj in range(len(i[x1][:-1])):
                        new.append(lowestcoor[x1][jj])
            flat.append(new)
        newpts = []
        for h2av in flat:
            coord = [[h2av[x], h2av[x+1], self.points[0][x//2][2]]
                     for x in range(0, len(h2av), 2)]
            newpts.append(coord)
        self.points = newpts

    def boundary(self, n, nr=0):
        """Apply periodic boundary conditions: n==1 to vertex `nr`, n==2 to all."""
        indices = [nr] if n == 1 else range(len(self.points))
        for count in indices:
            coord = self.points[count]
            for x1 in range(len(coord)):
                for ii in range(2):
                    halfbox = BoxDim[ii]/2
                    Z = coord[x1][ii]
                    coord[x1][ii] = Z - SignR(halfbox, Z) - SignR(halfbox, Z - BoxDim[ii])
            self.points[count] = coord


def Simplex_Reflection(simplex, highest, reflnr=2.0):
    """Reflect the highest vertex through the centroid of the others.

    Returns (reflected_config, contracted_config).
    """
    totalnr = nDim*nAtoms
    av     = [0.0]*totalnr    # centroid of the non-highest vertices
    h      = [0.0]*totalnr    # the highest vertex
    h2av   = [0.0]*totalnr    # reflected point
    alt2av = [0.0]*totalnr    # contracted point

    count = 0
    for i in simplex.points:
        target = av if count != highest else h
        tmp = 0
        for xx in i:
            for jj in xx[:-1]:            # x, y (skip charge)
                target[tmp] = target[tmp] + jj
                tmp = tmp + 1
        count += 1

    for i in range(len(av)):
        av[i] = av[i]/float(count - 1)
    for i in range(len(av)):
        h2av[i]   = h[i] + reflnr*(av[i] - h[i])       # reflection / expansion
        alt2av[i] = h[i] + FracShrimp1*(av[i] - h[i])  # contraction

    coord  = [[h2av[x],   h2av[x+1],   simplex.points[0][x//2][2]] for x in range(0, len(h2av), 2)]
    coord2 = [[alt2av[x], alt2av[x+1], simplex.points[0][x//2][2]] for x in range(0, len(alt2av), 2)]
    return coord, coord2

## 5. Parameters

These are the same parameters exposed by the sliders/entry boxes of the original GUI.
Change any of them and re-run **this cell together with the Initialisation and Run cells just below** to explore their effect (or use *Kernel → Restart & Run All*). Values are in the
toy model's arbitrary units (lengths in box/canvas units, energies loosely in kcal/mol).

### System and its properties

| Parameter | Meaning | Typical value / range |
|---|---|---|
| `nDim` | number of spatial dimensions | 2 (fixed) |
| `nAtoms` | number of particles | 2–40 — must leave room to place all atoms, or initialisation fails |
| `Radius` | particle radius (drawn size, and default absolute charge) | 10–40 |
| `Rmin` | position of the LJ energy minimum | `2.24 * Radius` |
| `BoxDim` | box dimensions (periodic) | `[500, 500]` |
| `Epsilon` | LJ well depth | 1–100 |
| `Dielec` | dielectric constant (charge screening) | 1 (vacuum) – 80 (water) |
| `qat` | absolute charge per atom | defaults to `Radius` |
| `frac_neg` | fraction of negative charges | 0–1 |
| `CutOff` | non-bonded cutoff distance | 250 |

### Minimizer (simplex / Nelder–Mead)

| Parameter | Meaning | Typical value / range |
|---|---|---|
| `Simplex_step` | size of the initial simplex (larger explores more in high dimension) | 300 |
| `FracShrimp1` | contraction of the highest vertex toward the centroid | 0.5 |
| `FracShrimp2` | shrink of all vertices toward the lowest (smaller = gentler, avoids early collapse) | 0.4 |
| `FracExpend` | expansion factor after a successful reflection | 2.0 |
| `deltaE` | convergence criterion on the energy improvement | 0.001 |
| `n2conv` | number of sub-`deltaE` steps (cumulative) to declare convergence | 600 |
| `max_iter` | hard cap on the number of steps | 20000 |

In [ ]:
nDim    = 2               # number of spatial dimensions
nAtoms  = 20              # number of particles
Radius  = 25.0            # particle radius (must leave room to place all atoms)
Rmin    = 2.24 * Radius   # distance at which the LJ energy is minimal
BoxDim  = [500, 500]      # box dimensions
Epsilon = 25.0            # LJ well depth
Dielec  = 1.0             # dielectric constant
qat     = Radius          # atom absolute charge
frac_neg = 0.5            # fraction of negative charges
OverlapFr = 0.0           # fraction of overlap allowed when placing atoms
CutOff  = 250             # non-bonded cutoff
CutOffSquare = CutOff**2

# --- simplex (Nelder-Mead) controls (tuned for the 20-atom system) ---
Simplex_step = 300.0      # size of the initial simplex (large -> more exploration in 40-D)
FracShrimp1  = 0.5        # contraction of the highest vertex toward the centroid
FracShrimp2  = 0.4        # shrink of all vertices toward the lowest (gentle -> avoids early collapse)
FracExpend   = 2.0        # expansion factor after a successful reflection
deltaE       = 0.001      # convergence criterion on the energy improvement
n2conv       = 600        # number of sub-deltaE steps (cumulative) to declare convergence

Seed     = 100            # random number seed (reproducibility)
max_iter = 20000          # safety cap on the number of simplex steps

## 6. Initialisation

Generate random, non-overlapping starting positions and assign charges
(a fraction `frac_neg` negative, the rest positive).

In [ ]:
import sys

### generate random, non-overlapping coordinates ###
def InitConf(n, dim, radius, qat, frac_neg):
    seed(Seed)
    print("Initializing box, please wait...")
    tmp_coord = []
    i = 0
    ntrial = 0
    nneg = int(float(n) * frac_neg)
    npos = n - nneg

    # first atom (no overlap check)
    x = random()*(dim[0]-radius) + radius
    y = random()*(dim[1]-radius) + radius
    charge = -qat
    if npos == n:
        charge = qat
    i += 1
    tmp_coord.append([x, y, charge])

    # remaining negative charges
    while i < nneg:
        x = random()*(dim[0]-radius) + radius
        y = random()*(dim[1]-radius) + radius
        OVERLAP = 1
        for j in range(i):
            if dist(tmp_coord[j], [x, y]) < (1-OverlapFr)*2*radius:
                OVERLAP = 0
        if OVERLAP:
            tmp_coord.append([x, y, -qat])
            i += 1
        ntrial += 1
        if ntrial > 100000:
            print("initialisation failed -> reduce radius or number of atoms")
            sys.exit()

    # remaining positive charges
    while i < n:
        x = random()*(dim[0]-radius) + radius
        y = random()*(dim[1]-radius) + radius
        OVERLAP = 1
        for j in range(i):
            if dist(tmp_coord[j], [x, y]) < (1-OverlapFr)*2*radius:
                OVERLAP = 0
        if OVERLAP:
            tmp_coord.append([x, y, qat])
            i += 1
        ntrial += 1
        if ntrial > 100000:
            print("initialisation failed -> reduce radius or number of atoms")
            sys.exit()
    return tmp_coord


Atom_Coord = InitConf(nAtoms, BoxDim, Radius, qat, frac_neg)
Color = [charge_color(a[2], qat) for a in Atom_Coord]
print(f"Placed {len(Atom_Coord)} atoms.")

## 7. Run the minimization

This loop replaces the GUI's `Go` callback / Tkinter event loop. Each step follows a simple
**decision tree** (a variant of Nelder–Mead) that always acts on the current
**highest**-energy vertex and accepts a candidate only if it beats that highest energy
`HighEne` by more than `deltaE`:

1. **Reflect** the highest vertex through the centroid.
   * If it lowers the energy → accept it *(move `Refl`)*, then greedily try an **expansion**
     (a further step in the same direction) and keep it too if it lowers the energy again
     *(move `Re-E`)*.
2. Otherwise **contract** the highest vertex toward the centroid.
   * If that lowers the energy → accept it *(move `High`)*.
3. Otherwise **shrink** the whole simplex toward the current lowest vertex *(move `Lowe`)* —
   the fallback when no single-vertex move helped.

After each move the changed vertices are wrapped back into the periodic box (`boundary`), the
simplex is re-evaluated, and the current **best** configuration and its energy are recorded
(tagged with the move type). The run stops once the best energy has improved by less than
`deltaE` on `n2conv` steps — counted **cumulatively**, a proxy for *"the simplex has stopped
making progress"* — or after at most `max_iter` steps.

In [ ]:
def run_minimization(Atom_Coord):
    """Headless downhill-simplex minimization. Returns (Simp, trajectory, E_hist, move_hist)."""
    Simp = Simplex(Atom_Coord, Simplex_step)
    ConvCrit1 = deltaE
    nconv = 0

    simplex_energy, highest_nr, lowest_nr = Simp.energy()
    best = Simp.points[lowest_nr]
    traj      = [[list(a) for a in best]]
    E_hist    = [simplex_energy[lowest_nr]]
    move_hist = ["init"]

    print("step %6d  E= %.3f" % (0, simplex_energy[lowest_nr]))

    for step in range(1, max_iter+1):
        simplex_energy, highest_nr, lowest_nr = Simp.energy()
        HighEne = simplex_energy[highest_nr]
        LowEne  = simplex_energy[lowest_nr]

        New_Atom_Coord, Alt_Coord = Simplex_Reflection(Simp, highest_nr)   # reflnr = 2
        New_Ene = Calc_Ene(New_Atom_Coord, Epsilon, Rmin, Dielec, CutOffSquare, BoxDim)

        if (HighEne - New_Ene) > ConvCrit1:
            # reflection was successful
            movetype = "Refl"
            Simp.Update_Simplex(New_Atom_Coord, highest_nr)
            Simp.boundary(1, highest_nr)
            # try an additional expansion
            New_Atom_Coord2, _ = Simplex_Reflection(Simp, highest_nr, -FracExpend)
            New_Ene2 = Calc_Ene(New_Atom_Coord2, Epsilon, Rmin, Dielec, CutOffSquare, BoxDim)
            if (New_Ene - New_Ene2) > ConvCrit1:
                movetype = "Re-E"
                Simp.Update_Simplex(New_Atom_Coord2, highest_nr)
                Simp.boundary(1, highest_nr)
        else:
            # reflection failed -> try contracting the highest vertex
            New_Ene = Calc_Ene(Alt_Coord, Epsilon, Rmin, Dielec, CutOffSquare, BoxDim)
            if (HighEne - New_Ene) > ConvCrit1:
                movetype = "High"
                Simp.Update_Simplex(Alt_Coord, highest_nr)
                Simp.boundary(1, highest_nr)
            else:
                # nothing helped -> shrink the whole simplex toward the lowest vertex
                Simp.shrimp(lowest_nr)
                Simp.boundary(2)
                movetype = "Lowe"

        simplex_energy, highest_nr, lowest_nr = Simp.energy()
        best = Simp.points[lowest_nr]
        NewLowEne = simplex_energy[lowest_nr]

        traj.append([list(a) for a in best])
        E_hist.append(NewLowEne)
        move_hist.append(movetype)

        Ene_diff = LowEne - NewLowEne
        if step % 20 == 0:
            print("step %6d  E= %.3f  move= %s" % (step, NewLowEne, movetype))

        # convergence: cumulative count of sub-deltaE improvements
        if abs(Ene_diff) < ConvCrit1:
            nconv += 1
            if nconv > n2conv:
                print("convergence reached at step %d (E= %.3f)" % (step, NewLowEne))
                break

    return Simp, traj, E_hist, move_hist


Simp, traj, E_hist, move_hist = run_minimization(Atom_Coord)
print(f"\nDone in {len(traj)-1} steps. Final (best) E = {E_hist[-1]:.3f}")

## 8. Energy convergence

How the best (lowest) energy in the simplex evolves during the minimization.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(range(len(E_hist)), E_hist, lw=2)
ax.set_xlabel("simplex step")
ax.set_ylabel("best energy")
ax.set_title("Simplex (Nelder-Mead) energy minimization")
ax.grid(alpha=0.3)
plt.show()

## 9. Visualise the system

The best configuration at the start and end of the run, side by side.
White = positive charge, dark = negative.

In [ ]:
def draw_config(ax, coord, title):
    ax.set_xlim(0, BoxDim[0])
    ax.set_ylim(0, BoxDim[1])
    ax.set_aspect('equal')
    ax.set_facecolor("#ccddff")
    ax.set_title(title)
    ax.invert_yaxis()   # match the original canvas (y downwards)
    for a in coord:
        col = charge_color(a[2], qat)
        ax.add_patch(Circle((a[0], a[1]), Radius, facecolor=col, edgecolor="black", lw=0.8))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 5.5))
draw_config(ax1, traj[0],  f"Initial  (E = {E_hist[0]:.1f})")
draw_config(ax2, traj[-1], f"Minimized (E = {E_hist[-1]:.1f})")
plt.tight_layout()
plt.show()

## 10. Animation of the minimization

Replays the best configuration over the whole run. This reproduces the main canvas of the
original GUI (which also displayed the full simplex alongside it).
(To keep it light, only every few frames are shown — adjust `stride`.)

In [ ]:
stride = max(1, len(traj)//120)   # cap at ~120 frames
frames = list(range(0, len(traj), stride))

fig, ax = plt.subplots(figsize=(6, 6))
ax.set_xlim(0, BoxDim[0])
ax.set_ylim(0, BoxDim[1])
ax.set_aspect('equal')
ax.set_facecolor("#ccddff")
ax.invert_yaxis()

circles = [Circle((a[0], a[1]), Radius,
                  facecolor=charge_color(a[2], qat), edgecolor="black", lw=0.8)
           for a in traj[0]]
for c in circles:
    ax.add_patch(c)
title = ax.set_title("")

def update(frame_idx):
    f = frames[frame_idx]
    for c, a in zip(circles, traj[f]):
        c.center = (a[0], a[1])
    title.set_text(f"step {f}   E = {E_hist[f]:.1f}   ({move_hist[f]})")
    return circles + [title]

anim = FuncAnimation(fig, update, frames=len(frames), interval=80, blit=False)
plt.close(fig)   # avoid a duplicate static figure
anim

## 11. Comparison of the three minimizers

This notebook implements **the downhill-simplex (Nelder–Mead) minimizer**. The three companion notebooks
(`LJ-ELEC_EM-steepest_Py3`, `LJ-ELEC_EM-conjugate_Py3`, `simplex`) minimize the **same system**
— 20 particles with identical parameters and the same random seed (`Seed = 100`). Running each
**with its default parameters** gives:

| Method | Final energy | Steps |
|---|---|---|
| Steepest descent | ≈ −409 | ~1100 |
| Conjugate gradient | ≈ −383 | ~970 |
| Simplex (Nelder–Mead) | ≈ −187 | ~620 |

Take-aways:

* All three reach a **local** minimum — none is guaranteed to find the global minimum, and the
  result depends on the starting configuration (the random seed) and the path taken.
* **Steepest descent** lands in the deepest basin here; **conjugate gradient** takes fewer steps
  but, following a different path, settles in a slightly shallower minimum.
* The **simplex** is *derivative-free* (energy only, no forces) — simple and robust, but it scales
  poorly to this 40-dimensional search space (2 × 20 coordinates), so it converges to a much
  shallower minimum. A good illustration of why gradient-based methods dominate for smooth,
  high-dimensional problems.

*(Energies and step counts are for `Seed = 100` with each notebook's default parameters; other
seeds give different absolute numbers.)*